# `ripopt`: direct Python interface with JAX autodiff

This notebook walks through the `ripopt` Python package — a direct, callback-based Python interface to [ripopt](https://github.com/jkitchin/ripopt), a primal-dual interior-point NLP solver written in Rust.

The workflow mirrors [cyipopt](https://cyipopt.readthedocs.io/), but you write **only** `f(x)` and `g(x)` in JAX-traceable Python. Gradients, Jacobians, and the Lagrangian Hessian are generated automatically via `jax.grad`, `jax.jacfwd`, and `jax.hessian`.

See also:
- [`docs/user_guide.md`](../docs/user_guide.md) — task-oriented prose guide
- [`docs/api.md`](../docs/api.md) — strict reference

## Setup

Build the extension once with `maturin develop --release` from the `ripopt-py/` directory, then import:

In [1]:
import os, sys, warnings
warnings.filterwarnings("ignore", category=UserWarning)

# ripopt logs internal diagnostics (fallback attempts, per-iter KKT state)
# directly to stderr (fd 2), independent of `print_level`. Silence them
# here so the notebook output stays readable. In your own session, drop
# the wrapper or pass `options={'print_level': 5}` for solver chatter.
from contextlib import contextmanager

@contextmanager
def _quiet():
    saved = os.dup(2)
    null = os.open(os.devnull, os.O_WRONLY)
    os.dup2(null, 2)
    try:
        yield
    finally:
        os.dup2(saved, 2)
        os.close(saved)
        os.close(null)

import jax.numpy as jnp
import numpy as np

from ripopt import minimize as _minimize

def minimize(*args, **kwargs):
    with _quiet():
        return _minimize(*args, **kwargs)

## 1. Unconstrained: Rosenbrock

The simplest case. Pass a scalar JAX-traceable function and a starting point.

$$f(x) = (1 - x_0)^2 + 100\,(x_1 - x_0^2)^2$$

The optimum is at $x^* = (1, 1)$ with $f^* = 0$.

In [2]:
def rosenbrock(x):
    return (1.0 - x[0])**2 + 100.0 * (x[1] - x[0]**2)**2

res = minimize(rosenbrock, x0=[-1.2, 1.0])

print(f"x*      = {res.x}")
print(f"f(x*)   = {res.fun:.3e}")
print(f"status  = {res.status}")
print(f"iters   = {res.iterations}")
print(f"wall    = {res.wall_time_secs * 1e3:.2f} ms")

x*      = [1. 1.]
f(x*)   = 5.395e-21
status  = Optimal
iters   = 34
wall    = 1.79 ms


## 2. Box constraints

Bounds can be scalar (broadcast to all variables), per-variable arrays, or a scipy-style list of `(lb, ub)` pairs. Use `np.inf` / `-np.inf` for missing sides.

Here the unconstrained minimum of $(x_0 - 3)^2 + (x_1 + 1)^2$ is $(3, -1)$; projected onto $[0, 2]^2$ it becomes $(2, 0)$.

In [3]:
def quad(x):
    return (x[0] - 3.0)**2 + (x[1] + 1.0)**2

res = minimize(quad, x0=[0.5, 0.5], bounds=(0.0, 2.0))

print(f"x*    = {res.x}")
print(f"f(x*) = {res.fun}")
print(f"z_u   = {res.bound_multipliers_upper}  # multiplier on x <= 2")
print(f"z_l   = {res.bound_multipliers_lower}  # multiplier on x >= 0")

x*    = [2. 0.]
f(x*) = 2.0
z_u   = [0. 0.]  # multiplier on x <= 2
z_l   = [0. 0.]  # multiplier on x >= 0


## 3. HS071 — the canonical cyipopt example

$$
\begin{aligned}
\min_{x} \quad & x_0 x_3 (x_0 + x_1 + x_2) + x_2 \\
\text{s.t.} \quad & x_0 x_1 x_2 x_3 \ge 25 \\
& x_0^2 + x_1^2 + x_2^2 + x_3^2 = 40 \\
& 1 \le x_i \le 5, \quad i = 0,\ldots,3
\end{aligned}
$$

Multiple constraints are packed into a single vector-valued `g(x)` with elementwise `lb`/`ub`. Equality is encoded as `lb == ub`.

The known optimum is $x^* \approx (1.00000, 4.74300, 3.82115, 1.37941)$ with $f^* \approx 17.0140$.

In [4]:
def f(x):
    return x[0] * x[3] * (x[0] + x[1] + x[2]) + x[2]

def g(x):
    return jnp.array([
        x[0] * x[1] * x[2] * x[3],                    # >= 25
        x[0]**2 + x[1]**2 + x[2]**2 + x[3]**2,        # == 40
    ])

res = minimize(
    f,
    x0=[1.0, 5.0, 5.0, 1.0],
    bounds=(1.0, 5.0),
    constraints={"fun": g, "lb": [25.0, 40.0], "ub": [np.inf, 40.0]},
    options={"tol": 1e-8},
)

print(f"x*                     = {res.x}")
print(f"f(x*)                  = {res.fun:.6f}")
print(f"g(x*)                  = {res.constraint_values}")
print(f"constraint multipliers = {res.constraint_multipliers}")
print(f"iterations             = {res.iterations}")

x*                     = [1.00000009 4.74299949 3.8211502  1.37940814]
f(x*)                  = 17.014017
g(x*)                  = [25.00000018 40.        ]
constraint multipliers = [-0.55229361  0.16146848]
iterations             = 18


## 4. Options and tolerances

Options mirror ipopt's naming. The commonly-used keys are:

| key | meaning |
|---|---|
| `tol` | overall convergence tolerance |
| `max_iter` | maximum IPM iterations |
| `print_level` | `0` (silent) to `5` (per-iteration log) |
| `max_wall_time` | seconds, `0` = none |
| `hessian_approximation` | `"exact"` or `"limited-memory"` |

See [`docs/api.md`](../docs/api.md) for the full list. Unknown keys raise `ValueError`.

In [5]:
res = minimize(
    rosenbrock,
    x0=[-1.2, 1.0],
    options={"tol": 1e-12, "max_iter": 200, "print_level": 0},
)

print(f"f(x*)            = {res.fun:.3e}")
print(f"final primal inf = {res.final_primal_inf:.3e}")
print(f"final dual   inf = {res.final_dual_inf:.3e}")
print(f"final compl      = {res.final_compl:.3e}")

f(x*)            = 1.245e-30
final primal inf = 0.000e+00
final dual   inf = 0.000e+00
final compl      = 0.000e+00


## 5. L-BFGS when the Hessian is expensive

For problems where `jax.hessian` (which computes $n^2$ entries) becomes expensive, skip Hessian construction entirely:

```python
options={"hessian_approximation": "limited-memory"}
```

The wrapper never calls `jax.hessian` — you only need `f` and the constraint functions. Gradients and the Jacobian are still exact via JAX.

Below: a 50-dimensional chained Rosenbrock solved with L-BFGS.

In [6]:
def rosenbrock_nd(x):
    return jnp.sum(100.0 * (x[1:] - x[:-1]**2)**2 + (1.0 - x[:-1])**2)

n = 50
x0 = np.full(n, -1.0)

res = minimize(
    rosenbrock_nd,
    x0=x0,
    options={
        "hessian_approximation": "limited-memory",
        "tol": 1e-6,
        "max_iter": 500,
    },
)

print(f"n          = {n}")
print(f"f(x*)      = {res.fun:.3e}")
print(f"status     = {res.status}")
print(f"iterations = {res.iterations}")
print(f"wall       = {res.wall_time_secs * 1e3:.1f} ms")

n          = 50
f(x*)      = 5.554e-14
status     = Optimal
iterations = 55
wall       = 24.3 ms


## 6. Sparsity detection

By default, `minimize` tells ripopt that every Jacobian and Hessian entry is structurally nonzero. Safe and simple.

For problems with genuine sparsity, pass `sparsity="detect"`. The wrapper probes the Jacobian at `x0` and at `x0 + perturb`, and the Lagrangian Hessian with two random `(σ, λ)` samples, and takes the union of nonzero entries. With random multipliers, the probability of missing a structurally-nonzero entry is zero in theory.

Sparsity detection shrinks the KKT system that ripopt factors; it does **not** reduce the per-callback JAX work.

In [7]:
res_dense = minimize(
    f,
    x0=[1.0, 5.0, 5.0, 1.0],
    bounds=(1.0, 5.0),
    constraints={"fun": g, "lb": [25.0, 40.0], "ub": [np.inf, 40.0]},
    options={"tol": 1e-8},
    sparsity="dense",
)

res_detect = minimize(
    f,
    x0=[1.0, 5.0, 5.0, 1.0],
    bounds=(1.0, 5.0),
    constraints={"fun": g, "lb": [25.0, 40.0], "ub": [np.inf, 40.0]},
    options={"tol": 1e-8},
    sparsity="detect",
)

print(f"dense  : f = {res_dense.fun:.6f}, iters = {res_dense.iterations}, wall = {res_dense.wall_time_secs*1e3:.2f} ms")
print(f"detect : f = {res_detect.fun:.6f}, iters = {res_detect.iterations}, wall = {res_detect.wall_time_secs*1e3:.2f} ms")

np.testing.assert_allclose(res_dense.x, res_detect.x, atol=1e-6)
print("\nSolutions agree to 1e-6.")

dense  : f = 17.014017, iters = 18, wall = 38.97 ms
detect : f = 17.014017, iters = 18, wall = 38.93 ms

Solutions agree to 1e-6.


## 7. Multiple constraint blocks

Instead of packing every constraint into one `g`, you can pass a list of dicts. The wrapper concatenates them internally. Useful when constraints come from different parts of the model.

Same HS071 expressed as two blocks:

In [8]:
def product_constraint(x):
    return x[0] * x[1] * x[2] * x[3]

def norm_constraint(x):
    return x[0]**2 + x[1]**2 + x[2]**2 + x[3]**2

constraints = [
    {"fun": product_constraint, "lb": 25.0, "ub": np.inf},
    {"fun": norm_constraint,    "lb": 40.0, "ub": 40.0},
]

res = minimize(
    f,
    x0=[1.0, 5.0, 5.0, 1.0],
    bounds=(1.0, 5.0),
    constraints=constraints,
    options={"tol": 1e-8},
)

print(f"f(x*) = {res.fun:.6f}")
print(f"g(x*) = {res.constraint_values}")

f(x*) = 17.014017
g(x*) = [25.00000018 40.        ]


## The `OptimizeResult` dataclass

Every call to `minimize` returns an `OptimizeResult` with these fields:

| field | meaning |
|---|---|
| `x` | primal optimum, shape `(n,)` |
| `fun` | `f(x*)` |
| `status` | `Optimal`, `Infeasible`, `MaxIterations`, `NumericalError`, ... |
| `success` | `status == 'Optimal'` |
| `iterations` | IPM iterations used |
| `constraint_multipliers` | Lagrange multipliers for `g(x)`, shape `(m,)` |
| `bound_multipliers_lower`, `bound_multipliers_upper` | shape `(n,)` |
| `constraint_values` | `g(x*)`, shape `(m,)` |
| `wall_time_secs` | total ripopt wall time including callback time |
| `final_primal_inf`, `final_dual_inf`, `final_compl` | KKT residuals at the final iterate |

## Next steps

- [`docs/user_guide.md`](../docs/user_guide.md) — deeper dive on JAX tips, constraint patterns, and troubleshooting
- [`docs/api.md`](../docs/api.md) — strict API reference
- [`examples/bench_hs071.py`](bench_hs071.py) — timing benchmark
- [cyipopt tutorial](https://cyipopt.readthedocs.io/stable/tutorial.html) — the interface this package emulates